In [2]:
import openai
import os
# from langchain.chat_models import ChatOpenAI
from langchain_community.chat_models import ChatOpenAI

from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
# from langchain.vectorstores import FAISS
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OpenAIEmbeddings
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.callbacks.base import BaseCallbackHandler

# from dotenv import Dotenv

from RealtimeTTS import GTTSEngine, TextToAudioStream
from pydub import AudioSegment
from pydub.playback import play
import ffmpeg
import warnings
warnings.filterwarnings("ignore")

# OpenAI Api Key
# OPENAI_API_KEY = Dotenv(".env")['OPENAI_API_KEY']
def load_env(file_path):
    with open(file_path) as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                key, value = line.strip().split('=', 1)
                os.environ[key] = value

load_env('.env')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')  

# Parameter
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
LOCAL_VECTORSTORE = "faiss_index_datamodel_bigchunks"
OPENAI_LLM_MODEL = "gpt-4o"

# Audio configurations
# Path to ffmpeg and ffprobe
ffmpeg_path = "D:/TexttoStream/ffmpeg/bin/ffmpeg.exe"
ffprobe_path = "D:/TexttoStream/ffmpeg/bin/ffprobe.exe"
# Set environment variables for pydub
AudioSegment.converter = ffmpeg_path
AudioSegment.ffmpeg = ffmpeg_path
AudioSegment.ffprobe = ffprobe_path



In [10]:
from langchain.callbacks.base import BaseCallbackHandler
import re

class TokenStreamCallbackHandler(BaseCallbackHandler):
    def __init__(self):
        super().__init__()
        self.tokens = []
    
    def on_llm_new_token(self, token: str, **kwargs):
        # Add the token to the list
        self.tokens.append(token)

    def get_streamed_tokens(self):
        # Generate tokens one at a time, then clear the list
        while self.tokens:
            token = self.tokens.pop(0)
            yield ''.join(re.findall(r'[A-Za-z0-9\s]', token))  # Filter out unwanted characters


In [11]:
openai.api_key = OPENAI_API_KEY

# Load the embedding model
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL, 
                              openai_api_key=OPENAI_API_KEY)

# Load the vector database index
persisted_vectorstore = FAISS.load_local(os.path.join("PakGPT",LOCAL_VECTORSTORE), 
                                         embeddings, 
                                         allow_dangerous_deserialization=True)

# Chat history
class LimitedConversationBufferMemory(ConversationBufferMemory):
    def save_context(self, inputs, outputs):
        super().save_context(inputs, outputs)
        # Limit the chat history to the last 3 messages
        self.chat_memory.messages = self.chat_memory.messages[-3:]

# Use the custom memory class
memory = LimitedConversationBufferMemory(memory_key='chat_history',
                                        return_messages=True, 
                                        output_key='answer')

# Retriever
retriever = persisted_vectorstore.as_retriever(search_kwargs={"k":3})


## LLM
token_stream_handler = TokenStreamCallbackHandler()
llm = ChatOpenAI(
    model_name=OPENAI_LLM_MODEL,
    openai_api_key=OPENAI_API_KEY,
    temperature=0,
    max_tokens=2500,
    timeout=30,
    streaming=True,
    callbacks=[token_stream_handler]  # Pass in the custom streaming handler
)

# Set up the conversation chain
prompt_template = """
You are a legal expert on the Constitution of Pakistan. Answer questions based only on its content. Follow these guidelines:

1. Stay relevant: Answer only what is asked without assumptions.
2. Stick to the Constitution: Politely refuse questions beyond its scope.
3. Ensure accuracy: Verify answers with the Constitution.
4. Be clear: Explain legal terms simply.
5. No speculation: Provide facts only.
6. Response should be less than 50 words and should be easy.
7. If the user ask about the previous response from the LLM then don't use the context in that question but use only previous questions and answers.
8. If the user says good bye message then respond with a happing ending message.
9. Summarize the previous conversation.

Previous Conversations: {chat_history}

Constitution Context: {context}

Query: {question}

Answer:
"""
custom_prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question", "chat_history"])

conversation_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": custom_prompt}
)


In [12]:
engine = GTTSEngine()
stream = TextToAudioStream(engine)
import re

In [13]:
def write_conversation(question: str):
    # Start the conversation chain with the question
    conversation_chain({"question": question})
    
    # Stream tokens from the callback handler
    for token in token_stream_handler.get_streamed_tokens():
        yield token

In [15]:
question = "What are the fundamental right of Pakistanis?"
text_stream = write_conversation(question)
stream.feed(text_stream)
stream.play()
